# Assignment 4: Experiment Runner 
**Course:** Research Methods and Tools (SE-2418)  
**Authors:** Aitimbet Baknur, Toilybekov Assylkhan, Zhaskairat Ayaulym  

---


In [ ]:
# 1. Verify GPU availability
!nvidia-smi


In [ ]:
# 2. Install required Python packages
!pip install -q sentence-transformers datasets ragas langchain-ollama langchain-huggingface pandas matplotlib seaborn scipy tqdm


In [ ]:
# 3. Install Ollama and start background daemon
!sudo apt-get update -qq
!sudo apt-get install -y -qq pciutils
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests

print("Starting Ollama background daemon...")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

for i in range(15):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            print("Ollama server is active and responding.")
            break
    except Exception:
        time.sleep(2)


In [ ]:
# 4. Pull models: Llama 3.1 8B (generator) and Qwen 2.5 7B (evaluator)
!ollama pull llama3.1:8b
!ollama pull qwen2.5:7b
!ollama list


In [ ]:
# 5. Optional file upload helper (skip if files were dragged into the left panel)
import os
required_files = ["rag_filter_pipeline.py", "evaluate_ragas.py", "consolidated_analysis.py", "generate_report_artifacts.py"]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    print("Please upload the missing files:", missing)
    from google.colab import files
    uploaded = files.upload()
else:
    print("All 4 required scripts are present in the workspace.")


In [ ]:
# 6. Execute test set generation (N = 200 questions, control-m = 4, tau = -2.0)
os.makedirs("runs", exist_ok=True)
!python rag_filter_pipeline.py --subset techqa --split test --n 200 --tau -2.0 --control-m 4 --out runs/test.jsonl


In [ ]:
# 7. Execute RAGAS evaluation across all 600 records using independent judge qwen2.5:7b
!python evaluate_ragas.py runs/test.jsonl scores.csv --judge qwen2.5:7b


In [ ]:
# 8. Compute statistical tests and generate figures
os.makedirs("analysis_output", exist_ok=True)
os.makedirs("report_artifacts", exist_ok=True)
!python consolidated_analysis.py scores.csv analysis_output
!python generate_report_artifacts.py scores.csv runs/test.jsonl report_artifacts


In [ ]:
# 9. Extract candidate failure cases for qualitative error analysis (Appendix B)
import csv, json

scores = list(csv.DictReader(open("scores.csv", encoding="utf-8")))
runs = [json.loads(line) for line in open("runs/test.jsonl", encoding="utf-8") if "condition" in json.loads(line)]
run_map = {(r["qid"], r["condition"]): r for r in runs}

candidates = []
for s in scores:
    try:
        f_val = float(s["faithfulness"])
    except Exception:
        f_val = 1.0
    try:
        c_val = float(s["answer_correctness"])
    except Exception:
        c_val = 1.0
    if f_val < 1.0 or c_val < 0.6:
        r = run_map.get((s["qid"], s["condition"]), {})
        candidates.append({
            "qid": s["qid"],
            "condition": s["condition"],
            "faithfulness": s["faithfulness"],
            "answer_correctness": s["answer_correctness"],
            "question": r.get("question", ""),
            "reference": r.get("reference", ""),
            "generated_answer": r.get("answer", ""),
            "contexts_preview": " // ".join(r.get("contexts", []))[:400]
        })

print(f"Extracted {len(candidates)} candidate failure instances for Appendix B annotation.")
with open("error_analysis_candidates.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "qid", "condition", "faithfulness", "answer_correctness",
        "question", "reference", "generated_answer", "contexts_preview"
    ])
    writer.writeheader()
    writer.writerows(candidates)
print("error_analysis_candidates.csv saved successfully.")


In [ ]:
# 10. Create complete results archive and download
!zip -r assignment4_completed_results.zip runs scores.csv report_artifacts analysis_output error_analysis_candidates.csv
print("Results package created successfully.")

from google.colab import files
files.download("assignment4_completed_results.zip")
